# Notebook 4: Modelos Benchmark
## Comparación Experimental contra el Denoising Autoencoder

**Proyecto:** Detección de Anomalías y Cambios de Régimen — ADRs Colombianos  
**Activos:** Ecopetrol (EC), Bancolombia (CIB), Grupo Aval (AVAL), Tecnoglass (TGLS)  
**Periodo:** 2015-01-01 — 2024-12-31  
**Autor:** [Nombre]  
**Versión:** 1.0

---

### Estructura del Notebook

| Sección | Modelo | Paradigma |
|---|---|---|
| §1 | Configuración y pipeline compartido | — |
| §2 | Protocolo experimental y métricas | — |
| §3 | Benchmark 1 — Isolation Forest | No supervisado |
| §4 | Benchmark 2 — One-Class SVM | No supervisado |
| §5 | Benchmark 3 — LSTM Predictor | Supervisado (pseudo) |
| §6 | Benchmark 4 — GRU Predictor | Supervisado (pseudo) |
| §7 | Benchmark 5 — Random Forest | Supervisado |
| §8 | Benchmark 6 — XGBoost | Supervisado |
| §9 | Benchmark 7 — Z-Score Estadístico | Estadístico clásico |
| §10 | Comparación global y tabla final | — |

---

### **Principio de comparación justa**

Todos los modelos comparten **exactamente** el mismo pipeline:

```
Features idénticas    : [log_return, vol_21d, vol_zscore]
Partición idéntica    : Train 2015-2019 | Val 2020 | Test 2021-2024
Normalización idéntica: StandardScaler ajustado sólo en train
Umbral de detección   : calibrado sobre val en todos los casos
Métricas idénticas    : Precisión, Recall, F1, AUC-ROC sobre test
```

La única diferencia entre modelos es la arquitectura y el mecanismo de
generación del score de anomalía. Esto garantiza que las diferencias en
rendimiento sean atribuibles al modelo y no al pipeline.

---
## Sección 1 — Configuración y Pipeline Compartido

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os, time
import numpy as np
import pandas as pd
from scipy import stats

# Machine Learning clásico
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, roc_curve,
    average_precision_score
)

# XGBoost
import xgboost as xgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

# Datos
import yfinance as yf

# Visualización
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reproducibilidad
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11,
                     'axes.labelsize': 10})

print(f"scikit-learn : {__import__('sklearn').__version__}")
print(f"xgboost      : {xgb.__version__}")
print(f"TensorFlow   : {tf.__version__}")


In [ ]:
# ── Parámetros globales ───────────────────────────────────────────────────────
TICKERS      = ['EC', 'CIB', 'AVAL', 'TGLS']
NAMES        = {'EC': 'Ecopetrol', 'CIB': 'Bancolombia',
                'AVAL': 'Grupo Aval', 'TGLS': 'Tecnoglass'}
COLORS       = {'EC': '#1f77b4', 'CIB': '#ff7f0e',
                'AVAL': '#2ca02c', 'TGLS': '#d62728'}

TRAIN_START  = '2015-01-01';  TRAIN_END  = '2019-12-31'
VAL_START    = '2020-01-01';  VAL_END    = '2020-12-31'
TEST_START   = '2021-01-01';  TEST_END   = '2024-12-31'

SEQ_LEN      = 30
N_FEATURES   = 3
FEATURE_COLS = ['log_return', 'vol_21d', 'vol_zscore']
ROLL_WIN     = 21
THRESHOLD_P  = 95          # percentil de umbral (calibrado en NB3)

ANOMALY_PERIODS = [
    {'name': 'Crisis petróleo', 'start': '2015-07-01',
     'end': '2016-02-01', 'color': 'rgba(128,0,128,0.10)'},
    {'name': 'COVID-19',        'start': '2020-02-15',
     'end': '2020-05-01', 'color': 'rgba(220,20,60,0.10)'},
    {'name': 'Fed hikes',       'start': '2022-01-01',
     'end': '2022-12-31', 'color': 'rgba(255,140,0,0.10)'},
]

# Colores por modelo para las gráficas comparativas
MODEL_COLORS = {
    'DAE-LSTM':        '#1f77b4',
    'DAE-GRU':         '#aec7e8',
    'Isolation Forest':'#ff7f0e',
    'One-Class SVM':   '#ffbb78',
    'LSTM Predictor':  '#2ca02c',
    'GRU Predictor':   '#98df8a',
    'Random Forest':   '#d62728',
    'XGBoost':         '#ff9896',
    'Z-Score':         '#9467bd',
}

print("Parámetros globales cargados.")


In [ ]:
# ── Pipeline de features (idéntico al Notebook 2) ────────────────────────────
def build_features(ticker):
    raw = yf.download(ticker, start=TRAIN_START, end=TEST_END,
                      auto_adjust=True, progress=False)
    d = raw[['Open','High','Low','Close','Volume']].copy().dropna()
    d.index = pd.to_datetime(d.index)
    d['log_return'] = np.log(d['Close'] / d['Close'].shift(1))
    d['vol_21d']    = d['log_return'].rolling(ROLL_WIN).std() * np.sqrt(252)
    lv = np.log1p(d['Volume'])
    rs = lv.rolling(ROLL_WIN).std().replace(0, np.nan)
    d['vol_zscore'] = (lv - lv.rolling(ROLL_WIN).mean()) / rs
    return d[FEATURE_COLS].dropna()

def create_windows(arr, seq_len):
    return np.lib.stride_tricks.sliding_window_view(
        arr, window_shape=(seq_len, arr.shape[1])
    ).squeeze(1).copy()

def make_anomaly_labels(dates_idx, anomaly_periods):
    labels = pd.Series(0, index=dates_idx)
    for ap in anomaly_periods:
        mask = ((labels.index >= ap['start']) &
                (labels.index <= ap['end']))
        labels[mask] = 1
    return labels.values

def prepare_all(ticker):
    """
    Construye y retorna todos los artefactos necesarios
    para entrenar y evaluar cualquier modelo.

    Retorna
    -------
    dict con:
      raw_splits   : DataFrames de features sin escalar
      sc_splits    : arrays numpy escalados
      wins         : tensores (n_win, T, F) por split
      flat_wins    : tensores aplanados (n_win, T*F) — para modelos clásicos
      scaler       : StandardScaler ajustado en train
      dates        : índices de fecha por split
      y_labels     : etiquetas binarias de anomalía por split
    """
    feat_df = build_features(ticker)

    raw_splits = {
        'train': feat_df.loc[:TRAIN_END],
        'val':   feat_df.loc[VAL_START:VAL_END],
        'test':  feat_df.loc[TEST_START:TEST_END],
    }

    scaler = StandardScaler()
    scaler.fit(raw_splits['train'].values)
    sc_splits = {s: scaler.transform(df.values)
                 for s, df in raw_splits.items()}

    wins, flat_wins, dates, y_labels = {}, {}, {}, {}
    for s, arr in sc_splits.items():
        w = create_windows(arr, SEQ_LEN)
        wins[s]      = w
        flat_wins[s] = w.reshape(w.shape[0], -1)  # (n, T*F)
        dates[s]     = raw_splits[s].index[SEQ_LEN - 1:]
        y_labels[s]  = make_anomaly_labels(dates[s], ANOMALY_PERIODS)

    return {
        'raw': raw_splits, 'sc': sc_splits,
        'wins': wins, 'flat': flat_wins,
        'scaler': scaler, 'dates': dates,
        'y': y_labels
    }

# Preparar todos los activos
print("Construyendo pipeline de features para todos los activos...")
data = {t: prepare_all(t) for t in TICKERS}

TICKER_PILOT = 'EC'
print("\nShapes del activo piloto (EC):")
for split in ['train', 'val', 'test']:
    print(f"  {split:<8}  wins={data[TICKER_PILOT]['wins'][split].shape}  "
          f"flat={data[TICKER_PILOT]['flat'][split].shape}")


---
## Sección 2 — Protocolo Experimental y Métricas

### **2.1 Función de evaluación unificada**

Todos los modelos son evaluados por la misma función, que recibe
el score de anomalía continuo y las etiquetas binarias, y devuelve
el conjunto completo de métricas.

In [ ]:
# ── Registro central de resultados ───────────────────────────────────────────
RESULTS = {}   # RESULTS[model_name][ticker] = dict de métricas

def calibrate_threshold(scores_val, y_val, percentiles=None):
    """
    Busca el percentil óptimo del score de entrenamiento que maximiza
    F1 sobre el conjunto de validación.

    Parámetros
    ----------
    scores_val  : np.ndarray — scores sobre validación
    y_val       : np.ndarray — etiquetas binarias de validación
    percentiles : list[int] — percentiles a explorar

    Retorna
    -------
    best_p : int — percentil óptimo
    """
    if percentiles is None:
        percentiles = [85, 90, 93, 95, 97, 99]
    best_p, best_f1 = 95, -1
    for p in percentiles:
        tau   = np.percentile(scores_val, p)
        y_hat = (scores_val > tau).astype(int)
        f1    = f1_score(y_val, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1, best_p = f1, p
    return best_p


def evaluate_model(model_name, ticker,
                   scores_train, scores_val, scores_test,
                   y_val, y_test,
                   verbose=True):
    """
    Evalúa un modelo de detección de anomalías.

    Protocolo:
    1. El umbral τ se calibra sobre el conjunto de validación.
    2. Las métricas finales se calculan sobre el conjunto de test.
    3. Los resultados se almacenan en el registro RESULTS.

    Parámetros
    ----------
    model_name   : str
    ticker       : str
    scores_train : np.ndarray — scores de anomalía en train
    scores_val   : np.ndarray — scores de anomalía en val
    scores_test  : np.ndarray — scores de anomalía en test
    y_val        : np.ndarray — etiquetas binarias val
    y_test       : np.ndarray — etiquetas binarias test

    Retorna
    -------
    dict con métricas
    """
    # Alinear longitudes
    n_val  = min(len(scores_val),  len(y_val))
    n_test = min(len(scores_test), len(y_test))
    sv, yv = scores_val[:n_val],   y_val[:n_val]
    st, yt = scores_test[:n_test], y_test[:n_test]

    # Calibrar umbral sobre val
    best_p  = calibrate_threshold(sv, yv)
    tau     = np.percentile(scores_train, best_p)
    y_pred  = (st > tau).astype(int)

    prec = precision_score(yt, y_pred, zero_division=0)
    rec  = recall_score(yt,   y_pred,  zero_division=0)
    f1   = f1_score(yt,       y_pred,  zero_division=0)
    try:
        auc  = roc_auc_score(yt, st)
        ap   = average_precision_score(yt, st)
    except Exception:
        auc = ap = float('nan')

    tp = int(((y_pred==1)&(yt==1)).sum())
    fp = int(((y_pred==1)&(yt==0)).sum())
    fn = int(((y_pred==0)&(yt==1)).sum())
    tn = int(((y_pred==0)&(yt==0)).sum())

    result = {
        'model':     model_name,
        'ticker':    ticker,
        'tau_p':     best_p,
        'tau':       tau,
        'precision': prec,
        'recall':    rec,
        'f1':        f1,
        'auc_roc':   auc,
        'avg_prec':  ap,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'scores_train': scores_train,
        'scores_val':   sv,
        'scores_test':  st,
        'y_pred':       y_pred,
        'y_test':       yt,
    }

    if model_name not in RESULTS:
        RESULTS[model_name] = {}
    RESULTS[model_name][ticker] = result

    if verbose:
        print(f"  [{model_name}] [{NAMES[ticker]}]  "
              f"P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}  "
              f"AUC={auc:.3f}  τ=p{best_p}")
    return result


print("Funciones de protocolo experimental definidas.")


---
## Sección 3 — Benchmark 1: Isolation Forest

### **Justificación**

El Isolation Forest (Liu et al., 2008) es el algoritmo de detección de anomalías
no supervisado de referencia en aprendizaje automático clásico. Su principio
se basa en que las anomalías son más fácilmente aislables (requieren menos
particiones en un árbol de decisión aleatorio) que los puntos normales.

**Pertinencia para este problema:**
Es el comparador natural del DAE porque ambos son no supervisados y aprenden
exclusivamente sobre datos normales. La diferencia fundamental es que Isolation
Forest opera sobre vectores de features independientes (ignora la secuencia
temporal), mientras que el DAE explota la estructura dinámica de la ventana.

### **Arquitectura**

```
Entrada  : vector aplanado de la ventana (T × F = 30 × 3 = 90 dimensiones)
Proceso  : ensemble de n_estimators árboles de aislamiento
           Cada árbol: particiones aleatorias recursivas sobre features aleatorias
Score    : profundidad media de aislamiento normalizada ∈ (0, 1)
           Valores altos → anomalía (aislado rápidamente)
Umbral   : score > τ calibrado sobre validación
```

### **Hiperparámetros**

| Parámetro | Valor | Justificación |
|---|---|---|
| `n_estimators` | 200 | Suficiente para estabilizar el score medio |
| `max_samples` | 256 | Submuestreo que reduce varianza sin perder cobertura |
| `contamination` | 'auto' | No asumimos fracción de anomalías conocida |
| `random_state` | 42 | Reproducibilidad |

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| No supervisado — no requiere etiquetas | Ignora la secuencia temporal |
| Eficiente computacionalmente | Opera sobre features aplanadas (pierde estructura) |
| Robusto a outliers en entrenamiento | No modela dependencias entre features a lo largo del tiempo |
| Interpretable (profundidad de aislamiento) | Sensible a la dimensionalidad (90 features) |

In [ ]:
# ── Isolation Forest ─────────────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest

print("BENCHMARK 1 — ISOLATION FOREST")
print("=" * 55)

IF_PARAMS = {
    'n_estimators':  200,
    'max_samples':   256,
    'contamination': 'auto',
    'random_state':  SEED,
    'n_jobs':        -1,
}

for ticker in TICKERS:
    d = data[ticker]

    # Entrenamiento sobre ventanas aplanadas del conjunto de train
    iforest = IsolationForest(**IF_PARAMS)
    iforest.fit(d['flat']['train'])

    # Score: negativo del score de anomalía original (más alto = más anómalo)
    # decision_function retorna valores más negativos para anomalías
    score_train = -iforest.decision_function(d['flat']['train'])
    score_val   = -iforest.decision_function(d['flat']['val'])
    score_test  = -iforest.decision_function(d['flat']['test'])

    evaluate_model(
        'Isolation Forest', ticker,
        score_train, score_val, score_test,
        d['y']['val'], d['y']['test']
    )


In [ ]:
# ── Visualización: score IF en el tiempo (activo piloto) ─────────────────────
d_pilot = data[TICKER_PILOT]
res_if  = RESULTS['Isolation Forest'][TICKER_PILOT]

score_if_all = np.concatenate([
    res_if['scores_train'],
    res_if['scores_val'],
    res_if['scores_test']
])
dates_all = np.concatenate([
    d_pilot['dates']['train'][:len(res_if['scores_train'])],
    d_pilot['dates']['val'][:len(res_if['scores_val'])],
    d_pilot['dates']['test'][:len(res_if['scores_test'])]
])

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=False)

# Score IF completo
axes[0].plot(dates_all, score_if_all,
             color=MODEL_COLORS['Isolation Forest'],
             linewidth=0.8, alpha=0.8)
axes[0].axhline(res_if['tau'], color='red', linewidth=1.4,
                linestyle='--', label=f"τ (p{res_if['tau_p']}) = {res_if['tau']:.4f}")
for ap in ANOMALY_PERIODS:
    axes[0].axvspan(pd.Timestamp(ap['start']),
                    pd.Timestamp(ap['end']),
                    alpha=0.10, color='red')
axes[0].set_title(f'{NAMES[TICKER_PILOT]} — Isolation Forest: Score de Anomalía',
                  fontweight='bold')
axes[0].set_ylabel('Score IF')
axes[0].legend()

# Distribución del score: train vs test
axes[1].hist(res_if['scores_train'], bins=60, density=True,
             color='#1f77b4', alpha=0.6, label='Train (2015-2019)')
axes[1].hist(res_if['scores_test'],  bins=40, density=True,
             color='#d62728', alpha=0.6, label='Test (2021-2024)')
axes[1].axvline(res_if['tau'], color='black', linewidth=1.4,
                linestyle='--', label=f"τ = {res_if['tau']:.4f}")
axes[1].set_title('Distribución del Score IF por Split', fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_isolation_forest.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Sección 4 — Benchmark 2: One-Class SVM

### **Justificación**

El One-Class SVM (Schölkopf et al., 2001) aprende la frontera de decisión
que encierra la región de mayor densidad de los datos de entrenamiento en
el espacio de features transformado por un kernel. Los puntos fuera de
esta frontera se clasifican como anomalías.

**Pertinencia:** Es el método clásico de detección de anomalías basado en
margen para datos de entrenamiento de una sola clase, equivalente a un
clasificador entrenado sólo con la clase normal. Permite evaluar si las
anomalías están geométricamente separadas de lo normal en el espacio de features.

### **Arquitectura**

```
Entrada  : vector aplanado de la ventana (90 dimensiones)
Kernel   : RBF (Radial Basis Function) — γ = 'scale'
           K(x, x') = exp(-γ ||x - x'||²)
Proceso  : maximiza el margen respecto al origen en el espacio de features
Score    : distancia con signo a la frontera de decisión
           Negativo → anomalía, Positivo → normal
Umbral   : −score > τ calibrado sobre validación
```

### **Hiperparámetros**

| Parámetro | Valor | Justificación |
|---|---|---|
| `kernel` | 'rbf' | Captura fronteras no lineales en alta dimensión |
| `nu` | 0.05 | Fracción máxima esperada de outliers en train (5%) |
| `gamma` | 'scale' | γ = 1/(F × var(X)) — auto-adaptativo a la escala |

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| Frontera de decisión no lineal | Computacionalmente costoso (O(n²)) en datasets grandes |
| Teóricamente sólido (márgenes) | Muy sensible a la elección de `nu` y `gamma` |
| No asume distribución de los datos | Ignora la estructura temporal de la ventana |

In [ ]:
# ── One-Class SVM ────────────────────────────────────────────────────────────
from sklearn.svm import OneClassSVM

print("BENCHMARK 2 — ONE-CLASS SVM")
print("=" * 55)

# Para escalabilidad, usar submuestra del training si N > 3000
MAX_TRAIN_OC_SVM = 2000

OC_SVM_PARAMS = {
    'kernel': 'rbf',
    'nu':     0.05,
    'gamma':  'scale',
}

for ticker in TICKERS:
    d = data[ticker]

    X_tr = d['flat']['train']
    # Submuestra aleatoria si es necesario
    if len(X_tr) > MAX_TRAIN_OC_SVM:
        idx  = np.random.choice(len(X_tr), MAX_TRAIN_OC_SVM, replace=False)
        idx  = np.sort(idx)
        X_tr = X_tr[idx]

    oc_svm = OneClassSVM(**OC_SVM_PARAMS)
    oc_svm.fit(X_tr)

    # Score: negativo de la función de decisión
    # (valores más negativos = más anómalos)
    def svm_score(X):
        return -oc_svm.decision_function(X)

    score_train = svm_score(d['flat']['train'])
    score_val   = svm_score(d['flat']['val'])
    score_test  = svm_score(d['flat']['test'])

    evaluate_model(
        'One-Class SVM', ticker,
        score_train, score_val, score_test,
        d['y']['val'], d['y']['test']
    )


---
## Sección 5 — Benchmark 3: LSTM Predictor (Error de Predicción)

### **Justificación**

El LSTM Predictor es un modelo supervisado de predicción que aprende a
predecir el siguiente timestep a partir de los anteriores. El score de
anomalía es el error de predicción: si el modelo no puede predecir correctamente
el siguiente valor, el patrón es inusual.

**Diferencia conceptual con el DAE:**

| Dimensión | DAE | LSTM Predictor |
|---|---|---|
| Paradigma | Reconstrucción de ventana completa | Predicción del siguiente paso |
| Qué aprende | Representación latente de la secuencia | Dinámica autoregresiva |
| Score | MSE(input, reconstrucción) | MSE(y_real_{t+1}, y_pred_{t+1}) |
| Horizonte | Toda la ventana T | Sólo el siguiente timestep |

### **Arquitectura**

```
Entrada  : (batch, T-1, F) — todos los timesteps menos el último
Proceso  : LSTM(64) → LSTM(32) → Dense(F)
Salida   : (batch, F) — predicción del timestep T
Score    : MSE(x_T, x̂_T) — error puntual en el último paso
```

### **Hiperparámetros**

| Parámetro | Valor |
|---|---|
| Unidades LSTM | [64, 32] |
| Dense output | F = 3 (una predicción por feature) |
| Optimizador | Adam (lr = 1e-3) |
| Loss | MSE |
| Early stopping patience | 15 épocas |

In [ ]:
# ── LSTM Predictor ────────────────────────────────────────────────────────────
print("BENCHMARK 3 — LSTM PREDICTOR")
print("=" * 55)

def build_predictor(seq_len, n_features, lstm_units, cell='LSTM'):
    """
    Modelo de predicción de siguiente paso.
    Entrada: (B, T-1, F) → Salida: (B, F)
    """
    RNN = layers.LSTM if cell == 'LSTM' else layers.GRU

    inp = keras.Input(shape=(seq_len - 1, n_features))
    x   = RNN(lstm_units[0], return_sequences=True,
              name=f'{cell}_1')(inp)
    x   = layers.Dropout(0.20)(x)
    x   = RNN(lstm_units[1], return_sequences=False,
              name=f'{cell}_2')(x)
    x   = layers.Dropout(0.20)(x)
    out = layers.Dense(n_features, name='output')(x)

    model = Model(inp, out, name=f'{cell}_Predictor')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return model


def predictor_score(model, X_windows):
    """
    Calcula el MSE de predicción del último paso de cada ventana.
    X_windows shape: (n, T, F)
    """
    X_in  = X_windows[:, :-1, :].astype(np.float32)  # (n, T-1, F)
    y_true= X_windows[:, -1,  :].astype(np.float32)  # (n, F)
    y_hat = model.predict(X_in, batch_size=256, verbose=0)
    return np.mean((y_true - y_hat) ** 2, axis=1)    # (n,)


def train_predictor(model, X_train, X_val):
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=7, verbose=0),
    ]
    model.fit(
        X_train[:, :-1, :].astype(np.float32),
        X_train[:, -1,  :].astype(np.float32),
        validation_data=(
            X_val[:, :-1, :].astype(np.float32),
            X_val[:, -1,  :].astype(np.float32)
        ),
        batch_size=64, epochs=150,
        callbacks=cb, shuffle=True, verbose=0
    )
    return model


for ticker in TICKERS:
    d = data[ticker]
    m = build_predictor(SEQ_LEN, N_FEATURES, [64, 32], cell='LSTM')
    train_predictor(m, d['wins']['train'], d['wins']['val'])

    evaluate_model(
        'LSTM Predictor', ticker,
        predictor_score(m, d['wins']['train']),
        predictor_score(m, d['wins']['val']),
        predictor_score(m, d['wins']['test']),
        d['y']['val'], d['y']['test']
    )


---
## Sección 6 — Benchmark 4: GRU Predictor

### **Justificación**

El GRU Predictor replica exactamente la arquitectura del LSTM Predictor
pero utiliza celdas GRU. Su inclusión permite aislar el efecto del tipo
de celda recurrente sobre el rendimiento de detección, manteniendo
constante el paradigma (predicción de siguiente paso vs. reconstrucción).

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| Menor número de parámetros que LSTM | Mismas limitaciones de horizonte puntual |
| Entrenamiento más rápido | Score basado en error de predicción, no reconstrucción |
| Compuertas simplificadas — menos sobreajuste | No accede a información de toda la ventana |

In [ ]:
# ── GRU Predictor ─────────────────────────────────────────────────────────────
print("BENCHMARK 4 — GRU PREDICTOR")
print("=" * 55)

for ticker in TICKERS:
    d = data[ticker]
    m = build_predictor(SEQ_LEN, N_FEATURES, [64, 32], cell='GRU')
    train_predictor(m, d['wins']['train'], d['wins']['val'])

    evaluate_model(
        'GRU Predictor', ticker,
        predictor_score(m, d['wins']['train']),
        predictor_score(m, d['wins']['val']),
        predictor_score(m, d['wins']['test']),
        d['y']['val'], d['y']['test']
    )


---
## Sección 7 — Benchmark 5: Random Forest

### **Justificación**

El Random Forest se incluye como representante de los métodos supervisados
de ensamble. A diferencia de todos los modelos anteriores (no supervisados),
el Random Forest utiliza **etiquetas de anomalía** para entrenarse. Esto
constituye una ventaja injusta en términos de información disponible, por lo
que su comparación establece el **límite superior de referencia supervisado**.

**Construcción de etiquetas de entrenamiento:**
Las etiquetas se asignan sobre el conjunto de entrenamiento usando los periodos
de anomalía conocidos (crisis del petróleo 2015–2016). Sólo el 3–5% de las
ventanas de entrenamiento tienen etiqueta positiva.

**Implicación metodológica:**
Un DAE que supere al Random Forest en AUC-ROC demostraría que el aprendizaje
no supervisado puede igualar o superar a un método supervisado con etiquetas
parciales en un contexto de clase desbalanceada.

### **Arquitectura**

```
Entrada  : vector aplanado (90 dimensiones) + estadísticos agregados
Features : [media_r, std_r, max_r, min_r, media_vol, max_vol,
            media_zscore, max_zscore, skew_r, kurt_r] — 10 features
Proceso  : ensemble de 300 árboles con bootstrap
Score    : probabilidad de clase positiva (anomalía)
```

### **Hiperparámetros**

| Parámetro | Valor | Justificación |
|---|---|---|
| `n_estimators` | 300 | Suficiente para estabilizar OOB error |
| `max_depth` | 10 | Limita la profundidad para evitar sobreajuste |
| `class_weight` | 'balanced' | Compensa el desbalance de clases (5% anomalías) |
| `max_features` | 'sqrt' | Selección aleatoria de features por árbol |

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| Robusto a features no lineales e interacciones | Requiere etiquetas — ventaja injusta |
| Manejo nativo de desbalance de clases | Ignora la secuencia temporal |
| Interpretable (importancia de features) | Performance dependiente de calidad de etiquetas |

In [ ]:
# ── Ingeniería de features estadísticos para RF y XGBoost ───────────────────
def window_stats_features(windows):
    """
    Extrae estadísticos descriptivos de cada ventana para modelos clásicos.
    Entrada: (n, T, F) → Salida: (n, n_stats)

    Features extraídas por cada dimensión original:
      media, std, max, min, skewness, kurtosis, rango IQR
    Más features globales de la ventana.
    """
    n, T, F = windows.shape
    feats   = []

    for f in range(F):
        col = windows[:, :, f]           # (n, T)
        feats.append(col.mean(axis=1))   # media
        feats.append(col.std(axis=1))    # std
        feats.append(col.max(axis=1))    # máximo
        feats.append(col.min(axis=1))    # mínimo
        feats.append(col.max(axis=1) - col.min(axis=1))  # rango
        # Skewness y kurtosis por ventana
        feats.append(np.array([stats.skew(col[i])     for i in range(n)]))
        feats.append(np.array([stats.kurtosis(col[i]) for i in range(n)]))
        # Tendencia (coeficiente de regresión lineal)
        x_t = np.arange(T)
        slopes = np.array([np.polyfit(x_t, col[i], 1)[0] for i in range(n)])
        feats.append(slopes)
        # Número de cruces por cero
        crossings = np.array(
            [((col[i, :-1] * col[i, 1:]) < 0).sum() for i in range(n)]
        ).astype(float)
        feats.append(crossings)

    # También incluir las features aplanadas (toda la secuencia)
    return np.column_stack(feats)   # (n, F * 9)


# Construir features estadísticas para todos los activos y splits
print("Construyendo features estadísticas para RF y XGBoost...")
for ticker in TICKERS:
    data[ticker]['stats'] = {
        s: window_stats_features(data[ticker]['wins'][s])
        for s in ['train', 'val', 'test']
    }
n_stat_feats = data[TICKER_PILOT]['stats']['train'].shape[1]
print(f"Features estadísticas por ventana: {n_stat_feats}")


In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
print("BENCHMARK 5 — RANDOM FOREST")
print("=" * 55)
print("Nota: modelo supervisado — usa etiquetas de anomalía en entrenamiento.")
print()

RF_PARAMS = {
    'n_estimators': 300,
    'max_depth':    10,
    'class_weight': 'balanced',
    'max_features': 'sqrt',
    'random_state': SEED,
    'n_jobs':       -1,
    'oob_score':    True,
}

for ticker in TICKERS:
    d = data[ticker]

    X_tr = d['stats']['train']
    X_vl = d['stats']['val']
    X_te = d['stats']['test']

    # Etiquetas de entrenamiento (incluye crisis petróleo en train)
    y_tr = d['y']['train'][:len(X_tr)]

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr, y_tr)

    # Score = probabilidad de clase positiva (anomalía)
    score_train = rf.predict_proba(X_tr)[:, 1]
    score_val   = rf.predict_proba(X_vl)[:, 1]
    score_test  = rf.predict_proba(X_te)[:, 1]

    evaluate_model(
        'Random Forest', ticker,
        score_train, score_val, score_test,
        d['y']['val'][:len(X_vl)],
        d['y']['test'][:len(X_te)]
    )


In [ ]:
# ── Importancia de features — Random Forest ───────────────────────────────────
# Calculamos sobre EC para ilustrar
d_rf_pilot = data[TICKER_PILOT]
y_tr_pilot = d_rf_pilot['y']['train'][:len(d_rf_pilot['stats']['train'])]

rf_pilot = RandomForestClassifier(**RF_PARAMS)
rf_pilot.fit(d_rf_pilot['stats']['train'], y_tr_pilot)

importances = rf_pilot.feature_importances_

# Nombres de features estadísticas
stat_names_base = ['media', 'std', 'max', 'min', 'rango',
                   'skew', 'kurt', 'slope', 'crossings']
feat_names = [f'{f}_{s}'
              for f in FEATURE_COLS
              for s in stat_names_base]

# Top 20 más importantes
top_idx  = np.argsort(importances)[::-1][:20]
top_imp  = importances[top_idx]
top_names= [feat_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(20), top_imp[::-1], color='#d62728', alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel('Importancia (Gini)')
ax.set_title(f'{NAMES[TICKER_PILOT]} — Random Forest: Top 20 Features más Importantes',
             fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rf_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Sección 8 — Benchmark 6: XGBoost

### **Justificación**

XGBoost (Chen & Guestrin, 2016) es el método de boosting de árboles de decisión
de mayor rendimiento en competencias de datos tabulares. Se incluye como el
representante del estado del arte en métodos supervisados sobre features
estadísticas extraídas de ventanas temporales.

**Diferencias respecto al Random Forest:**
- Boosting secuencial vs. bagging paralelo: XGBoost construye árboles que
  corrigen los errores del árbol anterior, lo que puede producir mejores
  fronteras de decisión en zonas difíciles.
- Regularización L1/L2 explícita sobre los pesos de los árboles.
- Manejo eficiente del desbalance mediante `scale_pos_weight`.

### **Hiperparámetros**

| Parámetro | Valor | Justificación |
|---|---|---|
| `n_estimators` | 300 | Número de rondas de boosting |
| `max_depth` | 6 | Árboles de profundidad moderada — menos sobreajuste que RF |
| `learning_rate` | 0.05 | Tasa de aprendizaje conservadora |
| `subsample` | 0.8 | Submuestreo estocástico por árbol |
| `colsample_bytree` | 0.8 | Submuestreo de features por árbol |
| `scale_pos_weight` | ~19 | Compensa desbalance: (N_neg / N_pos) |
| `eval_metric` | 'aucpr' | Área bajo curva PR — métrica robusta a desbalance |

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| Estado del arte en datos tabulares | Requiere etiquetas — ventaja injusta |
| Regularización explícita y eficiente | Sensible a hiperparámetros |
| Manejo óptimo de desbalance de clases | No captura dependencias temporales de orden alto |

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
print("BENCHMARK 6 — XGBOOST")
print("=" * 55)
print("Nota: modelo supervisado — usa etiquetas de anomalía en entrenamiento.")
print()

for ticker in TICKERS:
    d    = data[ticker]
    X_tr = d['stats']['train']
    X_vl = d['stats']['val']
    X_te = d['stats']['test']
    y_tr = d['y']['train'][:len(X_tr)]

    # Calcular scale_pos_weight para desbalance
    n_neg  = (y_tr == 0).sum()
    n_pos  = (y_tr == 1).sum()
    spw    = n_neg / max(n_pos, 1)

    XGB_PARAMS = {
        'n_estimators':     300,
        'max_depth':        6,
        'learning_rate':    0.05,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': spw,
        'eval_metric':      'aucpr',
        'use_label_encoder':False,
        'random_state':     SEED,
        'n_jobs':           -1,
        'verbosity':        0,
    }

    xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, d['y']['val'][:len(X_vl)])],
        verbose=False
    )

    score_train = xgb_model.predict_proba(X_tr)[:, 1]
    score_val   = xgb_model.predict_proba(X_vl)[:, 1]
    score_test  = xgb_model.predict_proba(X_te)[:, 1]

    evaluate_model(
        'XGBoost', ticker,
        score_train, score_val, score_test,
        d['y']['val'][:len(X_vl)],
        d['y']['test'][:len(X_te)]
    )


---
## Sección 9 — Benchmark 7: Z-Score Estadístico Multivariado

### **Justificación**

El Z-Score estadístico es el método más simple posible de detección de anomalías:
un punto es anómalo si se aleja más de k desviaciones estándar de la media
histórica. Se incluye como **línea base mínima**: cualquier modelo más complejo
debe superar este benchmark para justificar su mayor complejidad.

**Versión multivariada:**
Se calcula la distancia de Mahalanobis de cada ventana respecto a la distribución
empírica del conjunto de entrenamiento, lo que generaliza el Z-Score a múltiples
dimensiones considerando las correlaciones entre features.

```
score_t = sqrt( (x_t - μ)ᵀ Σ⁻¹ (x_t - μ) )

donde:
  x_t = vector de estadísticos de la ventana (media de cada feature)
  μ   = media muestral del conjunto de entrenamiento
  Σ   = matriz de covarianza del conjunto de entrenamiento
```

### **Ventajas y limitaciones**

| Ventajas | Limitaciones |
|---|---|
| Completamente interpretable | Asume distribución gaussiana multivariada |
| Sin hiperparámetros de modelo | Sólo captura desviaciones de primera y segunda orden |
| Baseline estadístico sólido | No modela la estructura temporal de la ventana |

In [ ]:
# ── Z-Score Estadístico (Distancia de Mahalanobis) ───────────────────────────
print("BENCHMARK 7 — Z-SCORE ESTADÍSTICO (DISTANCIA DE MAHALANOBIS)")
print("=" * 62)

def mahalanobis_score(X_train_stats, X_query_stats):
    """
    Calcula la distancia de Mahalanobis de cada muestra de X_query
    respecto a la distribución estimada en X_train.

    Parámetros
    ----------
    X_train_stats : (n_train, d) — features estadísticos de entrenamiento
    X_query_stats : (n_query, d) — features estadísticos a evaluar

    Retorna
    -------
    distancias : (n_query,) — distancia de Mahalanobis por muestra
    """
    mu    = X_train_stats.mean(axis=0)
    cov   = np.cov(X_train_stats.T)

    # Regularización para invertibilidad
    cov  += np.eye(cov.shape[0]) * 1e-6

    try:
        cov_inv = np.linalg.inv(cov)
    except np.linalg.LinAlgError:
        cov_inv = np.linalg.pinv(cov)

    diff  = X_query_stats - mu                    # (n, d)
    dist  = np.sqrt(
        np.einsum('ij,jk,ik->i', diff, cov_inv, diff)
    )                                              # (n,)
    return dist


for ticker in TICKERS:
    d = data[ticker]

    # Usar sólo los 3 estadísticos básicos (media por feature) para
    # mantener la matriz de covarianza bien condicionada
    def mean_features(wins):
        return wins.mean(axis=1)    # (n, F) — media temporal por feature

    X_tr_mf = mean_features(d['wins']['train'])
    X_vl_mf = mean_features(d['wins']['val'])
    X_te_mf = mean_features(d['wins']['test'])

    score_train = mahalanobis_score(X_tr_mf, X_tr_mf)
    score_val   = mahalanobis_score(X_tr_mf, X_vl_mf)
    score_test  = mahalanobis_score(X_tr_mf, X_te_mf)

    evaluate_model(
        'Z-Score', ticker,
        score_train, score_val, score_test,
        d['y']['val'][:len(X_vl_mf)],
        d['y']['test'][:len(X_te_mf)]
    )


---
## Sección 10 — Comparación Global y Análisis de Resultados

### **Carga de resultados del DAE (Notebook 3)**

Los resultados del DAE-LSTM y DAE-GRU se incorporan al registro de comparación.
Si el Notebook 3 ya fue ejecutado y los modelos guardados están disponibles,
se cargan directamente. En caso contrario se re-entrenan aquí con los mismos
hiperparámetros.

In [ ]:
# ── Re-entrenamiento del DAE para completar el registro RESULTS ──────────────
from tensorflow.keras import layers as L

def build_dae(seq_len, n_feat, latent=16, units=[64,32],
              drop=0.20, cell='LSTM', name='DAE'):
    RNN = L.LSTM if cell == 'LSTM' else L.GRU
    inp = keras.Input(shape=(seq_len, n_feat), name='input')
    x   = RNN(units[0], return_sequences=True)(inp)
    x   = L.Dropout(drop)(x)
    x   = RNN(units[1], return_sequences=False)(x)
    x   = L.Dropout(drop)(x)
    z   = L.Dense(latent, activation='tanh', name='latent_z')(x)
    y   = L.RepeatVector(seq_len)(z)
    y   = RNN(units[1], return_sequences=True)(y)
    y   = L.Dropout(drop)(y)
    y   = RNN(units[0], return_sequences=True)(y)
    y   = L.Dropout(drop)(y)
    out = L.TimeDistributed(L.Dense(n_feat))(y)
    m   = Model(inp, out, name=name)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return m


def add_noise(X, sigma=0.05):
    return (X + np.random.normal(0, sigma, X.shape)).astype(np.float32)


def train_dae(model, Xtr, Xvl, sigma=0.05, verbose=0):
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=7, verbose=0),
    ]
    model.fit(
        add_noise(Xtr, sigma), Xtr.astype(np.float32),
        validation_data=(Xvl.astype(np.float32),
                         Xvl.astype(np.float32)),
        batch_size=64, epochs=150,
        callbacks=cb, shuffle=True, verbose=verbose
    )
    return model


def dae_mse(model, X):
    Xf  = X.astype(np.float32)
    Xh  = model.predict(Xf, batch_size=256, verbose=0)
    return np.mean((Xf - Xh) ** 2, axis=(1, 2))


print("Entrenando DAE-LSTM y DAE-GRU para todos los activos...")
for ticker in TICKERS:
    d = data[ticker]
    for cell, model_name in [('LSTM', 'DAE-LSTM'), ('GRU', 'DAE-GRU')]:
        m = build_dae(SEQ_LEN, N_FEATURES, cell=cell, name=f'{model_name}_{ticker}')
        train_dae(m, d['wins']['train'], d['wins']['val'])

        evaluate_model(
            model_name, ticker,
            dae_mse(m, d['wins']['train']),
            dae_mse(m, d['wins']['val']),
            dae_mse(m, d['wins']['test']),
            d['y']['val'], d['y']['test'],
            verbose=True
        )


In [ ]:
# ── Tabla comparativa completa ───────────────────────────────────────────────
MODEL_ORDER = ['DAE-LSTM', 'DAE-GRU',
               'Isolation Forest', 'One-Class SVM',
               'LSTM Predictor', 'GRU Predictor',
               'Random Forest', 'XGBoost',
               'Z-Score']

PARADIGM = {
    'DAE-LSTM':        'No supervisado',
    'DAE-GRU':         'No supervisado',
    'Isolation Forest':'No supervisado',
    'One-Class SVM':   'No supervisado',
    'LSTM Predictor':  'Semi-supervisado',
    'GRU Predictor':   'Semi-supervisado',
    'Random Forest':   'Supervisado',
    'XGBoost':         'Supervisado',
    'Z-Score':         'Estadístico',
}

rows = []
for model_name in MODEL_ORDER:
    if model_name not in RESULTS:
        continue
    for ticker in TICKERS:
        if ticker not in RESULTS[model_name]:
            continue
        r = RESULTS[model_name][ticker]
        rows.append({
            'Modelo':    model_name,
            'Paradigma': PARADIGM.get(model_name, '—'),
            'Activo':    NAMES[ticker],
            'Precisión': round(r['precision'], 4),
            'Recall':    round(r['recall'],    4),
            'F1':        round(r['f1'],        4),
            'AUC-ROC':   round(r['auc_roc'],   4),
            'Avg. Prec': round(r['avg_prec'],  4),
            'τ percentil': r['tau_p'],
        })

results_df = pd.DataFrame(rows)
print("TABLA COMPARATIVA — TODOS LOS MODELOS Y ACTIVOS")
print("=" * 90)
print(results_df.to_string(index=False))


In [ ]:
# ── Tabla resumen: media entre activos por modelo ─────────────────────────────
summary = (results_df
           .groupby(['Modelo', 'Paradigma'])[['Precisión','Recall','F1','AUC-ROC','Avg. Prec']]
           .mean()
           .round(4)
           .sort_values('F1', ascending=False)
           .reset_index())

print("RANKING DE MODELOS (media entre activos) — ordenado por F1")
print("=" * 78)
print(summary.to_string(index=False))


In [ ]:
# ── Gráfica de barras: F1 y AUC-ROC por modelo ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_ordered = summary['Modelo'].tolist()
f1_vals   = summary['F1'].tolist()
auc_vals  = summary['AUC-ROC'].tolist()
colors_bar= [MODEL_COLORS.get(m, '#7f7f7f') for m in models_ordered]

# F1
axes[0].barh(models_ordered, f1_vals, color=colors_bar, alpha=0.85,
             edgecolor='white', linewidth=0.6)
for i, v in enumerate(f1_vals):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
axes[0].set_xlabel('F1-Score')
axes[0].set_title('F1-Score (promedio entre activos)', fontweight='bold')
axes[0].set_xlim([0, 1.05])
axes[0].invert_yaxis()

# AUC-ROC
axes[1].barh(models_ordered, auc_vals, color=colors_bar, alpha=0.85,
             edgecolor='white', linewidth=0.6)
for i, v in enumerate(auc_vals):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
axes[1].set_xlabel('AUC-ROC')
axes[1].set_title('AUC-ROC (promedio entre activos)', fontweight='bold')
axes[1].set_xlim([0, 1.05])
axes[1].invert_yaxis()

# Leyenda de paradigmas
legend_patches = [
    mpatches.Patch(color='#1f77b4', label='No supervisado'),
    mpatches.Patch(color='#2ca02c', label='Semi-supervisado'),
    mpatches.Patch(color='#d62728', label='Supervisado'),
    mpatches.Patch(color='#9467bd', label='Estadístico'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.04))

plt.suptitle('Comparación de Modelos Benchmark — Detección de Anomalías',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_benchmark_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Curvas ROC comparativas — activo piloto ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for model_name in MODEL_ORDER:
    if model_name not in RESULTS:
        continue
    if TICKER_PILOT not in RESULTS[model_name]:
        continue

    r     = RESULTS[model_name][TICKER_PILOT]
    n     = min(len(r['scores_test']), len(r['y_test']))
    st    = r['scores_test'][:n]
    yt    = r['y_test'][:n]
    color = MODEL_COLORS.get(model_name, '#7f7f7f')

    try:
        # ROC
        fpr, tpr, _ = roc_curve(yt, st)
        auc_val     = roc_auc_score(yt, st)
        axes[0].plot(fpr, tpr, color=color, linewidth=1.5,
                     label=f'{model_name} ({auc_val:.3f})')

        # PR
        prec_c, rec_c, _ = precision_recall_curve(yt, st)
        ap_val            = average_precision_score(yt, st)
        axes[1].plot(rec_c, prec_c, color=color, linewidth=1.5,
                     label=f'{model_name} (AP={ap_val:.3f})')
    except Exception:
        pass

# Diagonal ROC
axes[0].plot([0,1],[0,1], 'k--', linewidth=0.8, alpha=0.5)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title(f'{NAMES[TICKER_PILOT]} — Curva ROC',
                  fontweight='bold')
axes[0].legend(fontsize=7, loc='lower right')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisión')
axes[1].set_title(f'{NAMES[TICKER_PILOT]} — Curva Precisión-Recall',
                  fontweight='bold')
axes[1].legend(fontsize=7, loc='upper right')

plt.suptitle('Curvas ROC y Precisión-Recall — Comparación de Modelos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_roc_pr_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Heatmap de F1 por modelo × activo ────────────────────────────────────────
pivot_f1 = results_df.pivot(index='Modelo', columns='Activo', values='F1')
pivot_f1 = pivot_f1.reindex(
    [m for m in MODEL_ORDER if m in pivot_f1.index]
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot_f1, ax=ax,
    cmap='RdYlGn', vmin=0, vmax=1,
    annot=True, fmt='.3f', linewidths=0.5,
    cbar_kws={'label': 'F1-Score'}
)
ax.set_title('F1-Score por Modelo × Activo (Test Set 2021-2024)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Activo')
ax.set_ylabel('Modelo')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('fig_f1_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Score temporal superpuesto: DAE vs Isolation Forest ──────────────────────
ticker_comp = 'EC'
models_comp = ['DAE-LSTM', 'Isolation Forest', 'Z-Score']

fig = make_subplots(
    rows=len(models_comp), cols=1,
    shared_xaxes=True,
    subplot_titles=[f'{m} — Score de Anomalía ({NAMES[ticker_comp]})'
                    for m in models_comp],
    vertical_spacing=0.06
)

for i, model_name in enumerate(models_comp, start=1):
    if model_name not in RESULTS or ticker_comp not in RESULTS[model_name]:
        continue

    r     = RESULTS[model_name][ticker_comp]
    d_t   = data[ticker_comp]
    dates = np.concatenate([
        d_t['dates']['train'][:len(r['scores_train'])],
        d_t['dates']['val'][:len(r['scores_val'])],
        d_t['dates']['test'][:len(r['scores_test'])]
    ])
    scores = np.concatenate([
        r['scores_train'], r['scores_val'], r['scores_test']
    ])

    # Normalizar scores a [0,1] para comparación visual
    s_min, s_max = scores.min(), scores.max()
    s_norm = (scores - s_min) / (s_max - s_min + 1e-10)

    color = MODEL_COLORS.get(model_name, '#7f7f7f')
    fig.add_trace(
        go.Scatter(x=dates, y=s_norm, mode='lines',
                   name=model_name,
                   line=dict(color=color, width=0.9)),
        row=i, col=1
    )
    fig.add_hline(y=(r['tau'] - s_min) / (s_max - s_min + 1e-10),
                  line_dash='dash', line_color='red',
                  line_width=1.0, row=i, col=1)

    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=8,
            row=i, col=1
        )
    fig.update_yaxes(title_text='Score (norm.)', row=i, col=1)

fig.update_layout(
    title_text=f'<b>Comparación de Scores Normalizados — {NAMES[ticker_comp]}</b>',
    height=280 * len(models_comp), width=1100,
    template='plotly_white'
)
fig.show()


---
## Análisis de Resultados y Conclusiones

### **Jerarquía de rendimiento esperada**

Basado en las características de cada modelo y la naturaleza del problema,
se espera la siguiente jerarquía en rendimiento de detección:

```
MAYOR RENDIMIENTO
│
│  Supervisados (con etiquetas)
│  ├── XGBoost          — mayor capacidad discriminativa, acceso a etiquetas
│  └── Random Forest    — robusto, maneja desbalance nativamente
│
│  Semi-supervisados (error de predicción)
│  ├── LSTM Predictor   — captura dinámica temporal, semi-supervisado
│  └── GRU Predictor    — similar al LSTM con menor complejidad
│
│  No supervisados — aprenden solo sobre normalidad
│  ├── DAE-LSTM         — reconstrucción de secuencia completa + memoria LSTM
│  ├── DAE-GRU          — reconstrucción con menor parametrización
│  └── Isolation Forest — sin estructura temporal, pero robusto
│
│  Estadístico
│  └── Z-Score          — baseline mínimo; sólo captura desviaciones de media
│
MENOR RENDIMIENTO
```

### **Interpretación crítica de la comparación**

La comparación directa de métricas entre modelos supervisados y no supervisados
requiere una interpretación cuidadosa:

- **El DAE compite en desigualdad de condiciones:** aprende sin etiquetas,
  mientras que Random Forest y XGBoost utilizan las crisis conocidas del
  periodo de entrenamiento como ejemplos positivos. Un DAE con F1 comparable
  al Random Forest demuestra una capacidad de generalización superior.

- **AUC-ROC es la métrica más equitativa:** al no depender del umbral,
  permite comparar la separabilidad del score de anomalía independientemente
  de las ventajas de calibración de los modelos supervisados.

- **El Z-Score establece la barra mínima:** cualquier modelo que no supere
  al Z-Score multivariado no justifica su complejidad adicional.

### **Riesgos de interpretación**

- **Etiquetas de referencia imprecisas:** los periodos de anomalía son
  definiciones macro que pueden no capturar todos los sub-periodos de
  estrés dentro de una crisis o fuera de ella.
- **Variabilidad por activo:** el ranking de modelos puede diferir entre
  activos dependiendo de la intensidad y el tipo de anomalías presentes.
- **Sobreajuste supervisado:** Random Forest y XGBoost pueden mostrar
  métricas artificialmente altas en validación si las crisis del training
  y del test tienen perfiles estadísticos similares (e.g., ambas son
  crisis macro con alta volatilidad).

---

### **Próximo paso**

`Notebook_05_Analisis_Final.ipynb` — análisis de errores, visualización
profunda de anomalías detectadas por cada modelo, y generación del reporte
ejecutivo de resultados del proyecto.